## Understanding the data

What one row means, what each column holds, and how the class label was produced.

In [1]:
import io
import re
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
# the ten answer columns, A1 to A10
QUESTIONS = [f"A{i}" for i in range(1, 11)]

### 1. Loading the raw files

The UCI files are in ARFF format. `scipy.io.arff` fails on them because some values are quoted with trailing spaces (e.g. `'Middle Eastern '`), but everything after the `@data` line is plain CSV, so it is easier to read that part with pandas. Missing values are written as `?`.

In [2]:
# ARFF = Attribute-Relation File Format. It is a text file used by UCI.
# Column names are at the top, the rows below are normal CSV
def read_arff(path):
    # latin-1 never fails on a strange character
    text = Path(path).read_text(encoding="latin-1")
    header, body = re.split("@data", text, flags=re.IGNORECASE)
    # each line that starts with @attribute has one column name
    cols = [line.split()[1] for line in header.splitlines()
            if line.lower().startswith("@attribute")]
    # "?" means the value is missing
    return pd.read_csv(io.StringIO(body), header=None, names=cols,
                       quotechar="'", na_values="?", skipinitialspace=True)


raw = {
    "toddler": pd.read_csv(RAW / "toddler/Toddler Autism dataset July 2018.csv"),
    "child": read_arff(RAW / "child/Autism-Child-Data.arff"),
    "adolescent": read_arff(RAW / "adolescent/Autism-Adolescent-Data.arff"),
    "adult": read_arff(RAW / "adult/Autism-Adult-Data.arff"),
    "toddler_saudi": pd.read_csv(RAW / "toddler_saudi/Autism Spectrum Disorder Screening Data for Toddlers in Saudi Arabia Data Set.csv"),
    # test.csv has no labels, so we only read train.csv
    "adult_kaggle": pd.read_csv(RAW / "adult_kaggle/train.csv", na_values="?"),
}

# how many rows and columns each dataset has
pd.DataFrame({name: df.shape for name, df in raw.items()}, index=["rows", "columns"]).T

,rows,columns
toddler,1054,19
child,292,21
adolescent,104,21
adult,704,21
toddler_saudi,506,17
adult_kaggle,800,22


In [3]:
raw["adult"].head()

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,gender,ethnicity,jundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,1,1,0,0,1,1,0,0,...,f,White-European,no,no,United States,no,6,18 and more,Self,NO
1,1,1,0,1,0,0,0,1,0,1,...,m,Latino,no,yes,Brazil,no,5,18 and more,Self,NO
2,1,1,0,1,1,0,1,1,1,1,...,m,Latino,yes,yes,Spain,no,8,18 and more,Parent,YES
3,1,1,0,1,0,0,1,1,0,1,...,f,White-European,no,yes,United States,no,6,18 and more,Self,NO
4,1,0,0,0,0,0,0,1,0,0,...,f,NaN,no,no,Egypt,no,2,18 and more,NaN,NO


In [4]:
raw["toddler"].head()

,Case_No,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age_Mons,Qchat-10-Score,Sex,Ethnicity,Jaundice,Family_mem_with_ASD,Who completed the test,Class/ASD Traits
0,1,0,0,0,0,0,0,1,1,0,1,28,3,f,middle eastern,yes,no,family member,No
1,2,1,1,0,0,0,1,1,0,0,0,36,4,m,White European,yes,no,family member,Yes
2,3,1,0,0,0,0,0,1,1,0,1,36,4,m,middle eastern,yes,no,family member,Yes
3,4,1,1,1,1,1,1,1,1,1,1,24,10,m,Hispanic,no,no,family member,Yes
4,5,1,1,0,1,1,1,1,1,1,1,20,9,f,White European,no,yes,family member,Yes


### 2. The screening questionnaires

**Q-CHAT-10 (toddlers).** A parent answers ten questions on a five-point scale. From the dataset description:

| Item | Question |
|---|---|
| A1 | Does your child look at you when you call his/her name? |
| A2 | How easy is it for you to get eye contact with your child? |
| A3 | Does your child point to indicate that s/he wants something? |
| A4 | Does your child point to share interest with you? |
| A5 | Does your child pretend? (e.g. care for dolls, talk on a toy phone) |
| A6 | Does your child follow where you're looking? |
| A7 | If someone in the family is visibly upset, does your child show signs of wanting to comfort them? |
| A8 | Would you describe your child's first words as: (typical … unusual) |
| A9 | Does your child use simple gestures? (e.g. wave goodbye) |
| A10 | Does your child stare at nothing with no apparent purpose? |

For A1–A9 the answers *Sometimes / Rarely / Never* score 1; for A10 *Always / Usually / Sometimes* score 1.

**AQ-10 (children, adolescents, adults).** Ten statements answered *definitely agree … definitely disagree*. Each answer is scored 1 if it points towards autistic traits. There are three versions and **the wording differs between them**, so `A5` in the child data is not the same question as `A5` in the adult data. The full item lists are in `app/questions.py`.

In every dataset the columns `A1`–`A10` already hold the scored 0/1 value, not the original answer.

### 3. Columns

The UCI files (and the Kaggle adult file, which copies their layout) contain the following. Some names are misspelt in the original data.

| Column | Meaning | Used as a feature? |
|---|---|---|
| `A1_Score` … `A10_Score` | scored answers (0/1) | yes |
| `age` | age in years (months for toddlers) | yes |
| `gender` | m / f | yes |
| `jundice` | born with jaundice | yes |
| `austim` | an immediate family member has autism / PDD | yes |
| `ethnicity` | self-reported | no – only for the bias check |
| `contry_of_res` | country of residence | no – too many values for this data size |
| `used_app_before` | used the app before | no – not related to the person |
| `relation` | who completed the test | no |
| `age_desc` | age band, constant within a file | no |
| `result` | **total questionnaire score** | **no – the label is computed from it** |
| `Class/ASD` | ASD traits YES / NO | target |

The toddler files use different names (`Age_Mons`, `Qchat-10-Score`, `Class/ASD Traits `, …); notebook 03 maps everything to one schema.

In [5]:
# which columns have missing values
missing = {name: df.isna().sum()[lambda s: s > 0] for name, df in raw.items()}
pd.concat(missing).to_frame("missing values")

missing values
child        age                     4
             ethnicity              43
             relation               43
adolescent   ethnicity               6
             relation                6
adult        age                     2
             ethnicity              95
             relation               95
adult_kaggle ethnicity             203
             relation               40

Missing values only appear in `age`, `ethnicity` and `relation`. Only `age` matters for modelling.

### 4. How the class label was made

The toddler description says:

> *"the class variable was assigned automatically based on the score obtained by the user while undergoing the screening process using the ASDTests app."*

In other words, `ASD traits = 1` whenever the sum of A1–A10 is above 3 (Q-CHAT-10) or above 6 (AQ-10). Let's check that this holds for every row.

In [6]:
# the two toddler files use a different name for the label column
label_col = {"toddler": "Class/ASD Traits ", "toddler_saudi": "Class"}
# a score higher than this number means ASD traits
cutoff = {"toddler": 3, "toddler_saudi": 3, "child": 6, "adolescent": 6, "adult": 6, "adult_kaggle": 6}

rows = []
for name, df in raw.items():
    # the answer columns are called A1..A10 or A1_Score..A10_Score
    answers = df[[c for c in df.columns if re.fullmatch(r"A\d+(_Score)?", c)]]
    label = df[label_col.get(name, "Class/ASD")].astype(str).str.strip().str.lower().isin(["yes", "1"])
    # add up the answers ourselves and check if that gives the same label
    rule = answers.sum(axis=1) > cutoff[name]
    rows.append({"dataset": name, "cut-off": cutoff[name],
                 "ASD traits": f"{label.mean():.0%}",
                 "label matches score rule": f"{(rule == label).mean():.1%}"})

pd.DataFrame(rows)

,dataset,cut-off,ASD traits,label matches score rule
0,toddler,3,69%,100.0%
1,child,6,48%,100.0%
2,adolescent,6,61%,100.0%
3,adult,6,27%,100.0%
4,toddler_saudi,3,67%,100.0%
5,adult_kaggle,6,20%,80.5%


In [7]:
# adults: label against total score. The label changes exactly at the cut-off
adult = raw["adult"]
score = adult[[f"A{i}_Score" for i in range(1, 11)]].sum(axis=1)
pd.crosstab(adult["Class/ASD"], score.rename("sum of A1–A10"))

sum of A1–A10,0,1,2,3,4,5,6,7,8,9,10
Class/ASD,,,,,,,,,,,
NO,14,33,74,110,131,83,70,0,0,0,0
YES,0,0,0,0,0,0,0,57,55,47,30


#### What this means

In the five Thabtah and Saudi datasets the label is **exactly** "score above the cut-off", with no exceptions. This has some consequences for the rest of the project:

* Removing the `result` / score column is necessary but **not enough** to stop leakage. The model can still add up the ten answers itself, and a logistic regression can represent that rule perfectly. Near 100 % accuracy is therefore expected and does not show that a model "detects autism".
* The task the models learn is *reproducing the questionnaire scoring rule*, not predicting a clinical diagnosis. All results are reported with this in mind.
* Under the true rule every question carries **equal weight**. Any "most important question" found later (RQ3) reflects how answers are spread in this sample, not clinical importance.

The `adult_kaggle` data is different. It comes from a Kaggle competition where the data were generated from the original adult set. Its label only matches the rule about 80 % of the time, which makes it a harder and more realistic test for RQ2.

### 5. Summary

* Four main datasets (1,054 / 292 / 104 / 704 rows) and two second-source datasets (506 / 800 rows).
* Ten binary answers, age, gender, jaundice and family history are the useful inputs.
* The score column is a direct copy of the label rule and must not be used.
* The label is a screening outcome, not a diagnosis.